# cv_r1 合成ノート — 学習済みモデルの即時試聴

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slp-hu/Style-Bert-VITS2/blob/layer-b-cadence-seq/colab/cv_r1_synth_colab.ipynb)

学習済みモデル（本番 / スモークどちらも）で**テキストから即座に合成して聴く**ための軽量ノート。
評価ノート（`cv_r1_eval_speaker.ipynb`）の重い評価パイプライン（全話者 x-vector / UTMOS）は走らせない。

- **データセットの展開は不要**（model_assets の config / style_vectors / safetensors だけで動く）
- 学習直後の同一セッションなら §0 は数秒で素通りし、すぐ合成できる。新規 VM でも環境構築 ~10 分
- スモークモデルの注意: **学習済み 30 話者以外を指定しても雑音同然**
  （emb_g はランダム初期化のまま更新されていない）。学習済み話者は §1 が自動で表示する。
  数百 step のモデルなので品質は「声が出る」レベルの動作確認用
- virtual 話者（複数話者の emb_g 混合・補間）も §3 で試せる


In [ ]:
# ===== §0-1 マウント・前提チェック【毎回・ランタイム再起動後も最初に実行】=====
# --- Colab カーネル preload 対策 ---------------------------------------------
# Colab はカーネルを主要ライブラリ preload 済みスナップショットから起動するため、
# ディスクの numpy を入れ替えても、カーネルには旧版が sys.modules に残り続ける
# （再起動でも直らない）。版がズレていたら preload を破棄してディスク版を読み直させる。
import sys, subprocess

def _purge_stale_numpy():
    disk = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                          capture_output=True, text=True).stdout.strip()
    k = sys.modules.get("numpy")
    if k is None or not disk or getattr(k, "__version__", "") == disk:
        return
    victims = [m for m in list(sys.modules) if m.split(".")[0] in
               ("numpy", "pandas", "scipy", "matplotlib", "mpl_toolkits", "matplotlib_inline",
                "pylab", "seaborn", "sklearn", "pyarrow", "PIL")]
    for m in victims:
        del sys.modules[m]
    import numpy as _np
    assert _np.__version__ == disk, f"★numpy 差し替え失敗: {_np.__version__}（ランタイム再起動でやり直す）"
    import numpy.random   # サブパッケージまで健全なことを確認
    try:
        get_ipython().run_line_magic("matplotlib", "inline")   # inline バックエンドを新 matplotlib に再結線
    except Exception:
        pass
    print(f"カーネル preload の numpy {getattr(k, '__version__', '?')} 系 {len(victims)} モジュールを破棄 → {disk} に差し替え")

_purge_stale_numpy()
# -------------------------------------------------------------------------------
from google.colab import drive
from pathlib import Path
import os, subprocess

DRIVE_BASE = Path("/content/drive/MyDrive/Style-Bert-VITS2")   # setup と同じ値（clone の置き場所）

def _drive_alive():
    """マウントの生死確認（stale mount だと listdir が OSError: Transport endpoint is not connected）"""
    try:
        next(iter(os.listdir("/content/drive/MyDrive")), None)
        return True
    except OSError:
        return False

drive.mount("/content/drive")
if not _drive_alive():
    print("★Drive マウントが切れている（Transport endpoint is not connected）→ 強制再マウント")
    subprocess.run(["fusermount", "-u", "/content/drive"], capture_output=True)
    drive.mount("/content/drive", force_remount=True)
    assert _drive_alive(), "★再マウント失敗 → ランタイム再起動してやり直す"
assert DRIVE_BASE.exists(), f"★Drive に fork clone が無い: {DRIVE_BASE}（先に cadence_train_setup_colab.ipynb を実行）"

# --- fork の更新を Drive clone に自動反映（ベストエフォート。失敗しても続行）---
# バッジで開くノートは常に GitHub の最新だが、コードは Drive clone のスナップショット。
# ここで揃えないと「新しいノート × 古いコード」の不整合が起きる。意図的に版を固定したい場合は False。
AUTO_PULL = True
if AUTO_PULL:
    try:
        _p = subprocess.run(["git", "-C", str(DRIVE_BASE), "pull", "--ff-only"],
                            capture_output=True, text=True, timeout=180)
        if _p.returncode == 0:
            print("git pull:", (_p.stdout.strip().splitlines() or ["?"])[-1])
        else:
            print("★git pull 失敗（そのまま続行。clone の手元変更やネットワークを確認）:",
                  (_p.stderr or "").strip()[-200:])
    except Exception as e:
        print("★git pull 例外（そのまま続行）:", e)
_h = subprocess.run(["git", "-C", str(DRIVE_BASE), "rev-parse", "--short", "HEAD"],
                    capture_output=True, text=True).stdout.strip()
print("clone commit:", _h)

os.chdir(DRIVE_BASE); print("cwd:", Path.cwd())   # 依存インストールは requirements.txt をここから読む（後段でローカルへ移る）

br = subprocess.run(["git","rev-parse","--abbrev-ref","HEAD"], capture_output=True, text=True).stdout.strip()
assert br == "layer-b-cadence-seq", f"★branch が違う: {br} → git checkout layer-b-cadence-seq"
print("branch:", br)

g = subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True)
assert g.returncode == 0 and "GPU" in g.stdout, "★GPU ランタイムでない → [ランタイム]→[ランタイムのタイプを変更]→GPU"
print(g.stdout.strip())


In [ ]:
# ===== §0-2 GPU 判定と依存インストール【毎セッション実行・再起動後も再実行（冪等）】=====
# GPU の compute capability で経路を自動分岐する:
#   ・sm_90 以下（T4/L4/A100 等）: requirements の pin どおり torch 2.3.1(cu121)。再起動不要。
#   ・sm_100 以上（Blackwell 系: RTX PRO 6000 = sm_120 等）: torch 2.3.1 は sm_90 までで非対応
#     （GPU forward で落ちる）→ torch 2.11.0+cu128 へ入替（07a 方式）。★入替後はランタイム再起動が必須。
# 再起動後にこのセルを再実行すると、入替をスキップして検証だけ行う。
import subprocess, sys, re, os
from pathlib import Path

def _run(cmd, stream=False):
    print("$", " ".join(cmd))
    if stream:   # 進捗をそのまま流す（★torch の数GB DL は -q だと無言=フリーズと誤認するため）
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout: print(line, end="")
        p.wait(); assert p.returncode == 0, f"失敗: {cmd}"
    else:
        r = subprocess.run(cmd, capture_output=True, text=True)
        print((r.stdout or "")[-1200:] or "(quiet)")
        if r.returncode != 0:
            print(r.stderr[-4000:]); raise SystemExit(f"失敗: {cmd}")

# --- GPU capability（torch を import せず nvidia-smi で判定するのが肝）---
cap = subprocess.run(["nvidia-smi","--query-gpu=compute_cap","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()
assert cap and cap[0], "★GPU が見えない（GPU ランタイムか確認）"
CAP = float(cap[0]); IS_BLACKWELL = CAP >= 10.0
print(f"compute capability: {cap[0]} (sm_{int(CAP*10)}) → 経路: "
      + ("Blackwell（torch 2.11+cu128 入替）" if IS_BLACKWELL else "標準（torch 2.3.1 のまま）"))

# --- 現在の torch を subprocess で確認（in-kernel import しない = 再起動不要性を保つ）---
q = subprocess.run([sys.executable,"-c","import torch;print(torch.__version__)"], capture_output=True, text=True)
TORCH_NOW = q.stdout.strip()
print("現在の torch:", TORCH_NOW or "(未導入)")
ALREADY_SWAPPED = IS_BLACKWELL and TORCH_NOW.startswith("2.11.")

if ALREADY_SWAPPED:
    print("→ 再起動後の再実行と判断: torch 入替済みのため install をスキップ")
else:
    # (1) requirements から faster-whisper を除外して install（07a 方式）。
    #     faster-whisper==0.10.1 が av==10.* をソースビルドしようとし Py3.12 で失敗するため。
    #     文字起こし用途で cv_r1（bert_gen/style_gen/train/eval）には不要。
    req = Path("requirements.txt"); req_train = Path("requirements_no_whisper.txt")
    _drop = re.compile(r"^(faster-whisper|av)==")   # ビルドで詰まるパッケージが増えたらここに追加（例: stable_ts）
    kept = [l for l in req.read_text(encoding="utf-8").splitlines() if not _drop.match(l.strip())]
    req_train.write_text("\n".join(kept) + "\n", encoding="utf-8")
    _run([sys.executable,"-m","pip","install","-q","-r",str(req_train)])

    # Colab 既載の torchvision 等は torch 2.11 用ビルドのまま残り、torch 2.3.1 環境では壊れている
    # （torch.library.register_fake は 2.4+）。SBV2 は不使用で、残っていると umap 等が import して
    # 落ちるため standard 経路でも外す（Blackwell 経路と同じ扱い）。
    _run([sys.executable,"-m","pip","uninstall","-y","-q",
          "torchvision","torchcodec","torchao","torchtune","torchdata"])

    # (2) Blackwell のみ: torch 2.11.0+cu128 へ入替（実績のある手順をそのまま踏む）
    if IS_BLACKWELL:
        _run([sys.executable,"-m","pip","uninstall","-y",
              "torch","torchaudio","torchvision","torchcodec","torchao","torchtune","torchdata"])
        _run([sys.executable,"-m","pip","install","torch==2.11.0","torchaudio==2.11.0",
              "--index-url","https://download.pytorch.org/whl/cu128"], stream=True)   # ★-q 禁止
        _run([sys.executable,"-m","pip","install","-q","soundfile"])
        _run([sys.executable,"-m","pip","uninstall","-y",
              "torchcodec","torchvision","torchao","torchtune","torchdata"])

    # (3) HF スタック固定（transformers 未ピン → Colab 既定の transformers 5.x が torch>=2.4 を要求して
    #     torch を無効化し、bert_gen が "AutoModelForMaskedLM requires PyTorch" で落ちる問題の恒久対策）
    _run([sys.executable,"-m","pip","install","-q",
          "transformers==4.41.2","huggingface_hub==0.23.5","tokenizers<0.20",
          "pytorch-lightning==2.2.5","torchmetrics<1.5","pyannote.audio==3.1.1",
          "scipy==1.13.1","numpy==1.26.4"])

# --- numpy 健全性チェック（混在インストールの検出と自動修復）---
# pip のダウングレードで旧 2.x の compiled .so（numpy/random/mtrand 等）が残ると、
# コアは import できるのにサブパッケージ初期化で "numpy.dtype size changed" になる。
_h = subprocess.run([sys.executable,"-c","import numpy.random, numpy; print(numpy.__version__)"],
                    capture_output=True, text=True)
if _h.returncode != 0:
    print("★numpy が混在状態（サブパッケージ初期化に失敗）→ 1.26.4 を強制再インストールで修復")
    print("  症状:", (_h.stderr or "").strip().splitlines()[-1] if _h.stderr else "?")
    _run([sys.executable,"-m","pip","install","-q","--force-reinstall","--no-deps",
          "--no-cache-dir","numpy==1.26.4"])
    _h = subprocess.run([sys.executable,"-c","import numpy.random, numpy; print(numpy.__version__)"],
                        capture_output=True, text=True)
    assert _h.returncode == 0, "★numpy を修復できない:\n" + (_h.stderr or "")[-600:]
print("numpy 健全性 OK:", _h.stdout.strip())

# --- 検証（別プロセス。transformers から torch が見えているかまで確認）---
v = subprocess.run([sys.executable,"-c",
    "import torch;from transformers.utils import is_torch_available;"
    "print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),"
    "'| transformers_sees_torch',is_torch_available())"], capture_output=True, text=True)
print(v.stdout.strip() or v.stderr[-800:])
assert "transformers_sees_torch True" in v.stdout, "★transformers が torch を認識していない → このセルをやり直す"
if (not IS_BLACKWELL) or ALREADY_SWAPPED:
    assert "cuda True" in v.stdout, "★CUDA が使えない（GPU ランタイム / torch ビルドを確認）"

# --- カーネル整合性チェック: カーネルに import 済みの numpy とディスクの numpy が食い違うと、
#     このノートの in-kernel import（torch/numba 等）が "numpy.dtype size changed" で落ちる。
#     （Colab 素の numpy 2.x をカーネルが先に読み込み、上の install が 1.26.4 へ下げた場合に起きる）
_disk_np = subprocess.run([sys.executable,"-c","import numpy;print(numpy.__version__)"],
                          capture_output=True, text=True).stdout.strip()
_knp = sys.modules.get("numpy")
NUMPY_MISMATCH = _knp is not None and getattr(_knp, "__version__", "") != _disk_np

if NUMPY_MISMATCH:
    # Colab のカーネル preload（snapshot）由来のズレは再起動では直らない → その場で差し替える
    print(f"カーネルの numpy {_knp.__version__} ≠ ディスク {_disk_np} → preload を破棄して差し替え")
    for _m in [x for x in list(sys.modules) if x.split(".")[0] in
               ("numpy", "pandas", "scipy", "matplotlib", "mpl_toolkits", "matplotlib_inline",
                "pylab", "seaborn", "sklearn", "pyarrow", "PIL")]:
        del sys.modules[_m]
    import numpy as _np_chk
    assert _np_chk.__version__ == _disk_np, f"★差し替え失敗: {_np_chk.__version__}（ランタイム再起動でやり直す）"
    import numpy.random
    try:
        get_ipython().run_line_magic("matplotlib", "inline")
    except Exception:
        pass
    print("カーネル numpy:", _np_chk.__version__, "（再起動不要）")

if IS_BLACKWELL and not ALREADY_SWAPPED:
    print()
    print("=" * 70)
    print("★torch を入れ替えた → ここで【ランタイム再起動】が必須★")
    print("  [ランタイム] → [セッションを再起動] のあと、先頭セルから順に再実行してから先へ進む。")
    print("  （再実行時は install が即完了し、この警告は出なくなる）")
    print("=" * 70)
else:
    print("環境 OK → 次のセルへ")


In [ ]:
# ===== §0-3 torchaudio / torch.load shim【Blackwell 経路のみ実体化・毎セッション実行】=====
# torch 2.11 系では torchaudio.set_audio_backend が削除され、torchaudio.load も torchcodec 経由で壊れる。
# pyannote.audio が import 時に set_audio_backend を呼ぶため、shim なしでは style_gen / 合成が落ちる。
# `!python` で走る bert_gen / style_gen / train は【別プロセス】なので、カーネル内 monkeypatch では効かない
# → sitecustomize.py + PYTHONPATH で全 Python プロセスに注入する（ここが肝）。
import os, subprocess, sys
from pathlib import Path

if not IS_BLACKWELL:
    print("標準経路（torch 2.3.1）: shim 不要 → スキップ")
else:
    compat = Path("/content/_compat"); compat.mkdir(exist_ok=True)
    shim = compat / "sitecustomize.py"
    SHIM_SRC = '# sitecustomize: torch 2.11 環境の互換 shim（cv_r1 公開ノート用）\n# 1) torch.load の weights_only 既定を False に戻す（旧 ckpt / torch.hub モデルの読込互換）\n# 2) torchaudio.set_audio_backend / get_audio_backend を復活（pyannote.audio の import 時呼び出し対策）\n# 3) torchaudio.load / info を soundfile 実装に置換（torchcodec 経由の破綻を回避）\ntry:\n    import torch\n    _orig_torch_load = torch.load\n    def _patched_torch_load(*args, **kwargs):\n        kwargs.setdefault("weights_only", False)\n        return _orig_torch_load(*args, **kwargs)\n    torch.load = _patched_torch_load\nexcept Exception:\n    pass\n\ntry:\n    import torchaudio\n\n    def _noop_set_backend(*args, **kwargs):\n        return None\n    def _get_backend(*args, **kwargs):\n        return "soundfile"\n    torchaudio.set_audio_backend = _noop_set_backend\n    torchaudio.get_audio_backend = _get_backend\n\n    def _sf_load(filepath, frame_offset=0, num_frames=-1, normalize=True,\n                 channels_first=True, format=None, buffer_size=4096, backend=None):\n        import soundfile as sf\n        import torch as _torch\n        frames = int(num_frames) if int(num_frames) > 0 else -1\n        data, sr = sf.read(str(filepath), start=int(frame_offset), frames=frames,\n                           dtype="float32", always_2d=True)\n        wav = _torch.from_numpy(data.T if channels_first else data)\n        return wav, sr\n    torchaudio.load = _sf_load\n\n    def _sf_info(filepath, format=None, buffer_size=4096, backend=None):\n        import soundfile as sf\n        info = sf.info(str(filepath))\n        class _AudioMetaData:\n            pass\n        meta = _AudioMetaData()\n        meta.sample_rate = info.samplerate\n        meta.num_frames = info.frames\n        meta.num_channels = info.channels\n        meta.bits_per_sample = 16\n        meta.encoding = "PCM_S"\n        return meta\n    torchaudio.info = _sf_info\nexcept Exception:\n    pass\n'
    shim.write_text(SHIM_SRC, encoding="utf-8")

    # 書き出し事故（末尾のエスケープ崩れ等 → SyntaxError）を必ず py_compile で検証する
    r = subprocess.run([sys.executable,"-m","py_compile",str(shim)], capture_output=True, text=True)
    assert r.returncode == 0, "★shim が SyntaxError: " + r.stderr[-600:]

    pp = os.environ.get("PYTHONPATH","")
    if str(compat) not in pp.split(":"):
        os.environ["PYTHONPATH"] = f"{compat}:{pp}" if pp else str(compat)
    print("PYTHONPATH:", os.environ["PYTHONPATH"])

    # 機能確認: 別プロセスで torchaudio.load が shim（_compat）実装に置換されているか
    chk = subprocess.run([sys.executable,"-c",
        "import torchaudio;torchaudio.set_audio_backend('soundfile');"
        "import inspect;print('shim OK:', inspect.getsourcefile(torchaudio.load))"],
        capture_output=True, text=True, env=os.environ.copy())
    print(chk.stdout.strip() or chk.stderr[-800:])
    assert "shim OK" in chk.stdout and "_compat" in chk.stdout, "★shim が別プロセスに効いていない"


In [ ]:
# ===== §0-4 コード配置 + model_assets 接続（データセット展開は不要）=====
import os
from pathlib import Path
ROOT = Path("/content/Style-Bert-VITS2")
DRIVE_BASE = Path("/content/drive/MyDrive/Style-Bert-VITS2")
ROOT.mkdir(parents=True, exist_ok=True)
# fork のコード一式を Drive clone からローカルへ（データ・成果物ツリーは除外）
!rsync -a --exclude Data --exclude .git --exclude model_assets --exclude eval_out {DRIVE_BASE}/ {ROOT}/
os.chdir(ROOT)
!ln -sfn {DRIVE_BASE}/model_assets {ROOT}/model_assets
print("cwd:", Path.cwd())
print("model_assets ->", os.readlink(ROOT/"model_assets"))
print("§0-4 完了（コード = Drive clone の rsync / 合成に必要なデータは model_assets のみ）")


In [ ]:
# ===== §1 モデル選択・ロード =====
import os, sys, re, json
BASE = "/content/Style-Bert-VITS2"
os.chdir(BASE); sys.path.insert(0, BASE)
from pathlib import Path
import numpy as np, torch

TARGET = "auto"   # "full" | "smoke" | "auto"（full があれば full、無ければ smoke）

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_all  = list(Path("model_assets").glob("*/*_e*_s*.safetensors"))
_full  = [p for p in _all if p.parent.name != "cv_r1_smoke"]
_smoke = [p for p in _all if p.parent.name == "cv_r1_smoke"]
_pick = {"full": _full, "smoke": _smoke}.get(TARGET) or _full or _smoke
assert _pick, "★model_assets に safetensors が無い（学習を完走させるか、§0-4 の symlink を確認）"
CKPT = max(_pick, key=lambda p: int(re.search(r"_s(\d+)\.safetensors$", p.name).group(1)))
ASSETS = CKPT.parent; IS_SMOKE = (ASSETS.name == "cv_r1_smoke")
CONFIG, SV_PATH = ASSETS/"config.json", ASSETS/"style_vectors.npy"
print(("スモーク" if IS_SMOKE else "本番"), "モデル:", CKPT.name)
assert CONFIG.exists(),  f"★config 無し: {CONFIG}（学習開始時に default_style が生成するはず）"
assert SV_PATH.exists(), f"★style_vectors 無し: {SV_PATH}"

from style_bert_vits2.tts_model import TTSModel
from style_bert_vits2.nlp import bert_models
from style_bert_vits2.constants import Languages
bert_models.load_model(Languages.JP, "ku-nlp/deberta-v2-large-japanese-char-wwm")
bert_models.load_tokenizer(Languages.JP, "ku-nlp/deberta-v2-large-japanese-char-wwm")

model = TTSModel(model_path=Path(CKPT), config_path=Path(CONFIG),
                 style_vec_path=Path(SV_PATH), device=DEVICE)
model.load(); net_g = model.net_g; net_g.eval()
hps = model.hyper_parameters
NEUTRAL = model.get_style_vector(0, 1.0)
spk2id  = hps.data.spk2id
SR      = hps.data.sampling_rate

# 話者候補: trained_speakers.json（スモーク学習が書き出す）→ esd_train.list → 全話者
tsj = ASSETS/"trained_speakers.json"
if tsj.exists():
    SPEAKERS = json.load(open(tsj, encoding="utf-8"))
elif Path("Data/cv_r1/esd_train.list").exists():
    SPEAKERS = sorted({l.split("|")[1] for l in open("Data/cv_r1/esd_train.list", encoding="utf-8")})
else:
    SPEAKERS = sorted(spk2id)
    if IS_SMOKE:
        print("★注意: 学習済み話者リストが見つからないため全 298 話者を表示中。"
              "スモークで学習していない話者は雑音同然になる")
sd = net_g.state_dict()
cad = {k: round(float(sd[k].float().norm()), 4) for k in sd if "cadence_cond" in k and "weight" in k}
print("cadence_cond ‖w‖:", cad, "（スモークでは小さくて正常）")
print(f"loaded | 話者候補 {len(SPEAKERS)}: {SPEAKERS[:8]}{' ...' if len(SPEAKERS) > 8 else ''} | SR {SR} | {DEVICE}")


In [ ]:
# ===== §2 単一話者で合成（自由テキスト。実行するたびここだけ書き換えて再実行）=====
from style_bert_vits2.models.infer import get_text
from IPython.display import display, Audio, Markdown

TEXT    = "音声合成のテストです。今日はとても良い天気ですね。"
SPEAKER = SPEAKERS[0]        # ← §1 が表示した話者候補から選ぶ（例: "cv_0001"）

@torch.no_grad()
def synth_text(text, spk, sdp_ratio=0.2, noise_scale=0.667, noise_scale_w=0.8, length_scale=1.0):
    """自由テキスト合成（g2p は推論時に実行。cadence 非注入 = emb_g の弾き分けのみ）"""
    net_g.cadence_weight_dp = net_g.cadence_weight_sdp = 0.0
    bert, ja_bert, en_bert, phone, tone, lang = get_text(text, Languages.JP, hps, DEVICE)
    out = net_g.infer(
        phone.to(DEVICE).unsqueeze(0),
        torch.LongTensor([phone.size(0)]).to(DEVICE),
        torch.LongTensor([spk2id[spk]]).to(DEVICE),
        tone.to(DEVICE).unsqueeze(0), lang.to(DEVICE).unsqueeze(0),
        ja_bert.to(DEVICE).unsqueeze(0),
        style_vec=torch.from_numpy(NEUTRAL).to(DEVICE).unsqueeze(0), cadence_vec=None,
        sdp_ratio=sdp_ratio, noise_scale=noise_scale,
        noise_scale_w=noise_scale_w, length_scale=length_scale)
    return out[0][0, 0].data.cpu().float()

w = synth_text(TEXT, SPEAKER)
display(Markdown(f"**{SPEAKER}**: {TEXT}")); display(Audio(w, rate=SR, normalize=True))


In [ ]:
# ===== §2.5 元話者の音声を聴く（合成との比較用）=====
SPK_REF = SPEAKER   # ← 聴きたい話者（§2 と同じでも別でも）

def find_original(spk):
    """優先順: 配置済みデータ（train/smoke セッション）の esd → model_assets の参照クリップ"""
    for esd in (Path("Data/cv_r1/esd_train.list"), Path("Data/cv_r1/esd_val.list")):
        if esd.exists():
            for l in open(esd, encoding="utf-8"):
                f = l.rstrip("\n").split("|")
                if f[1] == spk:
                    rel = f[0].split("Data/cv_r1/", 1)[1] if "Data/cv_r1/" in f[0] else f[0]
                    return Path("Data/cv_r1")/rel, (f[3] if len(f) > 3 else "")
    for rd in (ASSETS/"reference_clips", ASSETS.parent/"reference_clips"):   # モデル直下 → 共有置き場
        rj = rd/"reference_clips.json"
        if rj.exists():
            m = json.load(open(rj, encoding="utf-8")).get(spk)
            if m:
                return rd/m["file"], m.get("text", "")
    return None, None

_p, _t = find_original(SPK_REF)
if _p is None:
    print(f"★{SPK_REF} の元音声が見つからない: データ未配置かつ参照クリップ未同梱。\n"
          "  → demo/make_reference_clips.py で生成して model_assets/<model>/reference_clips/ に置く")
else:
    display(Markdown(f"**元音声 {SPK_REF}**: {_t}")); display(Audio(str(_p)))


In [ ]:
# ===== §2.6（任意）元音声ソースの取得 — §2.5 の音源が無いときだけ仕事をする =====
# 優先順: 参照クリップ or 配置済みデータがあれば何もしない（§2.5 がそのまま鳴る）。
# 無い場合、スモークモデルなら専用バンドル（~0.3 GB）を取得して配置する。
# 本番モデルで音源が無い場合は eval §9 で参照クリップを一度生成するのが正道（全量 DL はしない）。
import shutil
from pathlib import Path

_clips = any((d / "reference_clips.json").exists()
             for d in (ASSETS / "reference_clips", ASSETS.parent / "reference_clips"))
_data = Path("Data/cv_r1/esd_train.list").exists()

if _clips or _data:
    print("元音声ソースあり（" + ("参照クリップ" if _clips else "配置済みデータ") + "）→ このセルは何もしない。§2.5 をそのまま使う")
elif IS_SMOKE:
    URL = ("https://github.com/slp-hu/Style-Bert-VITS2/releases/download/"
           "cv_r1-smoke-v1/cadence_cv_r1_smoke_v1.tgz")
    !wget -q -O /content/_smoke.tgz "{URL}"
    assert Path("/content/_smoke.tgz").stat().st_size > 100_000_000, "★バンドル DL 失敗"
    stage = Path("/content/_smoke_x")
    if stage.exists(): shutil.rmtree(stage)
    stage.mkdir()
    !tar -xf /content/_smoke.tgz -C {stage}
    cfgs = list(stage.rglob("config.json")); assert len(cfgs) == 1, cfgs
    dst = Path("Data/cv_r1"); dst.parent.mkdir(exist_ok=True)
    if dst.exists() and not dst.is_symlink(): shutil.rmtree(dst)
    shutil.move(str(cfgs[0].parent), str(dst))
    print("Data/cv_r1 配置完了（30 話者分の元音声）→ §2.5 を再実行")
else:
    print("★元音声ソースが無い（本番モデル・クリップ未生成）:\n"
          "  eval ノートの §9 を一度実行して model_assets/reference_clips を生成する")


In [ ]:
# ===== §3 virtual 話者（emb_g 混合・補間）=====
# 実在しない声を作る: 複数話者の emb_g を重み付き平均して合成（eval ノート §7 と同一方式）
MIX  = {SPEAKERS[0]: 0.5, SPEAKERS[1]: 0.5}   # ← 混合比は自由（正規化は自動）
TEXT3 = TEXT   # §2 の文を流用（変えても良い）

# --- まず構成話者それぞれの元音声を聴く（素材A・素材B・混合結果の3点で比較する）---
if "find_original" in globals():
    for _s, _w in MIX.items():
        _p0, _t0 = find_original(_s)
        if _p0 is not None:
            display(Markdown(f"**元音声 {_s}**（混合比 {_w:g}）: {_t0}")); display(Audio(str(_p0)))
        else:
            print(f"（{_s} の元音声ソースなし — §2.5/§2.6 参照）")
else:
    print("（構成話者の元音声を聴くには §2.5 を先に実行しておく）")

@torch.no_grad()
def synth_with_embg(text, ref_spk, g_vec):
    emb_w = net_g.emb_g.weight.data
    slot = spk2id[ref_spk]; backup = emb_w[slot].clone()
    emb_w[slot] = g_vec.to(emb_w.device, emb_w.dtype)
    try:
        return synth_text(text, ref_spk)
    finally:
        emb_w[slot] = backup

emb = net_g.emb_g.weight.data
def gvec(spk): return emb[spk2id[spk]].clone()

tot = sum(MIX.values())
g = sum((w0/tot) * gvec(s) for s, w0 in MIX.items())
w = synth_with_embg(TEXT3, next(iter(MIX)), g)
display(Markdown(f"**混合** {MIX}: {TEXT3}")); display(Audio(w, rate=SR, normalize=True))

# 2話者の補間を聴き比べたいとき（コメントを外す）:
# A, B = gvec(SPEAKERS[0]), gvec(SPEAKERS[1])
# for a in (0.0, 0.5, 1.0):
#     w = synth_with_embg(TEXT3, SPEAKERS[0], (1-a)*A + a*B)
#     display(Markdown(f"α={a}")); display(Audio(w, rate=SR, normalize=True))
